# Notebook 12 — Corrected V3 Biometric Privacy Re-evaluation

**One-click Colab `Runtime → Run all`** (Python 3.12) for the **existing** 120 AffectNet+ originals and 600 generated images from the immutable Notebook 07 experiment. Reuses the frozen N10/N11 V3 detector scale (512→320 content, centred in a 640×640 zero canvas), verified SCRFD ONNX detector, and ArcFace/AuraFace recognizers. No generation or hyperparameter selection.

All 600 source–generated pairs receive row-level detector, landmark, embedding and cosine-similarity outcomes, with failures retained, **never counted as privacy success**. Visual review and identity correctness remain `pending`, so similarity scores are *provisional technical estimates*, not validated re-identification rates. The existing N07 emotion/utility fields are copied as historical comparison only. No face-match threshold or real re-identification risk is inferred.

Writes only into a unique `nb12_...` run directory in Google Drive. Historic notebooks, images and outputs are strictly read-only. A Colab disconnect may interrupt execution; source-level persistent checkpoints permit recovery only through an explicit fresh compatible run, not automatic claim of success.

## 1 — Frozen configuration
Reuse the same pinned, verified private Python environment and models. The V3 configuration is immutable, threshold 0.10, deterministic CPU. No interactive flags.

In [ ]:
import json, hashlib, os, sys, platform, subprocess, tempfile, time, uuid, csv
from pathlib import Path
from datetime import datetime, timezone
CONFIG = json.loads("{\n  \"schema_version\": 1,\n  \"method_id\": \"nb10_detector_scale_canvas_v1\",\n  \"classification\": \"exploratory_scientific_validity_diagnostic\",\n  \"parent07\": \"20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664\",\n  \"parent08\": \"20261013_nb08_biometric-diagnostic_48369686_3f72a02e_20261013T181313795949Z_eb90117f\",\n  \"sources\": [\n    {\n      \"source_id\": \"source_000\",\n      \"emotion\": \"neutral\",\n      \"historical_stratum\": \"success\"\n    },\n    {\n      \"source_id\": \"source_001\",\n      \"emotion\": \"neutral\",\n      \"historical_stratum\": \"faceanalysis_failed\"\n    },\n    {\n      \"source_id\": \"source_005\",\n      \"emotion\": \"happy\",\n      \"historical_stratum\": \"success\"\n    },\n    {\n      \"source_id\": \"source_008\",\n      \"emotion\": \"happy\",\n      \"historical_stratum\": \"faceanalysis_failed\"\n    },\n    {\n      \"source_id\": \"source_013\",\n      \"emotion\": \"sad\",\n      \"historical_stratum\": \"success\"\n    },\n    {\n      \"source_id\": \"source_010\",\n      \"emotion\": \"sad\",\n      \"historical_stratum\": \"faceanalysis_failed\"\n    },\n    {\n      \"source_id\": \"source_017\",\n      \"emotion\": \"surprise\",\n      \"historical_stratum\": \"success\"\n    },\n    {\n      \"source_id\": \"source_015\",\n      \"emotion\": \"surprise\",\n      \"historical_stratum\": \"faceanalysis_failed\"\n    },\n    {\n      \"source_id\": \"source_021\",\n      \"emotion\": \"fear\",\n      \"historical_stratum\": \"success\"\n    },\n    {\n      \"source_id\": \"source_020\",\n      \"emotion\": \"fear\",\n      \"historical_stratum\": \"faceanalysis_failed\"\n    },\n    {\n      \"source_id\": \"source_026\",\n      \"emotion\": \"angry\",\n      \"historical_stratum\": \"success\"\n    },\n    {\n      \"source_id\": \"source_025\",\n      \"emotion\": \"angry\",\n      \"historical_stratum\": \"faceanalysis_failed\"\n    },\n    {\n      \"source_id\": \"source_044\",\n      \"emotion\": \"neutral\",\n      \"historical_stratum\": \"yolo_failed\"\n    },\n    {\n      \"source_id\": \"source_019\",\n      \"emotion\": \"surprise\",\n      \"historical_stratum\": \"yolo_failed\"\n    }\n  ],\n  \"priority\": [\n    \"source_008\",\n    \"source_015\",\n    \"source_017\",\n    \"source_000\",\n    \"source_013\",\n    \"source_005\",\n    \"source_021\",\n    \"source_026\"\n  ],\n  \"references\": {\n    \"R1\": \"Recomputed N08 A: RGB ndarray YOLO largest box/margin15; compare saved prepared hash; FaceAnalysis 1.0.1 det1024 thresh0.1\",\n    \"R2\": \"Minimal upstream FaceAnalysis full RGB prepared image -> BGR; same models, det1024 thresh0.1; direct detect/get/trace comparison\",\n    \"R3\": \"Independent YOLO face graph on full PIL RGB input, confidence0.25; diagnostic only\",\n    \"R4\": \"Conditional legacy 0.7.3 decoder replay of identical captured raw tensors only if independent decode disagrees with native coordinates\"\n  },\n  \"det_size\": [\n    1024,\n    1024\n  ],\n  \"det_thresh\": 0.1,\n  \"yolo_confidences\": [\n    0.25,\n    0.1,\n    0.05\n  ],\n  \"crop_margin\": 0.15,\n  \"seed\": 42,\n  \"providers\": [\n    \"CPUExecutionProvider\"\n  ],\n  \"device\": \"cpu\",\n  \"models\": {\n    \"arcface\": {\n      \"pack\": \"buffalo_l\",\n      \"archive_url\": \"https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip\",\n      \"onnx_sha256\": {\n        \"1k3d68.onnx\": \"df5c06b8a0c12e422b2ed8947b8869faa4105387f199c477af038aa01f9a45cc\",\n        \"2d106det.onnx\": \"f001b856447c413801ef5c42091ed0cd516fcd21f2d6b79635b1e733a7109dbf\",\n        \"det_10g.onnx\": \"5838f7fe053675b1c7a08b633df49e7af5495cee0493c7dcf6697200b85b5b91\",\n        \"genderage.onnx\": \"4fde69b1c810857b88c64a335084f1c3fe8f01246c9a191b48c7bb756d6652fb\",\n        \"w600k_r50.onnx\": \"4c06341c33c2ca1f86781dab0e829f88ad5b64be9fba56e56bc9ebdefc619e43\"\n      }\n    },\n    \"auraface\": {\n      \"repo\": \"fal/AuraFace-v1\",\n      \"revision\": \"686774cad65e40933022896195e07e01ee06bee9\",\n      \"onnx_sha256\": {\n        \"1k3d68.onnx\": \"df5c06b8a0c12e422b2ed8947b8869faa4105387f199c477af038aa01f9a45cc\",\n        \"2d106det.onnx\": \"f001b856447c413801ef5c42091ed0cd516fcd21f2d6b79635b1e733a7109dbf\",\n        \"genderage.onnx\": \"4fde69b1c810857b88c64a335084f1c3fe8f01246c9a191b48c7bb756d6652fb\",\n        \"glintr100.onnx\": \"a7933ea5330113b01c9b60351d8f4c33003f145d8470ac5f0e52ee2effe25c60\",\n        \"scrfd_10g_bnkps.onnx\": \"5838f7fe053675b1c7a08b633df49e7af5495cee0493c7dcf6697200b85b5b91\"\n      }\n    },\n    \"yolo\": {\n      \"repo\": \"Bingsu/adetailer\",\n      \"revision\": \"53cc19de382014514d9d4038601d261a7faa9b7b\",\n      \"filename\": \"face_yolov8n.pt\",\n      \"sha256\": \"70b640f8f60b1cf0dcc72f30caf3da9495eb2fb6509da48c53374ad6806e6a9c\"\n    }\n  },\n  \"pins\": {\n    \"numpy\": \"1.26.4\",\n    \"pandas\": \"2.2.2\",\n    \"Pillow\": \"11.3.0\",\n    \"matplotlib\": \"3.10.0\",\n    \"onnx\": \"1.17.0\",\n    \"onnxruntime\": \"1.19.2\",\n    \"insightface\": \"1.0.1\",\n    \"ultralytics\": \"8.4.121\",\n    \"opencv-python\": \"4.11.0.86\",\n    \"huggingface-hub\": \"0.36.2\",\n    \"scipy\": \"1.15.3\",\n    \"scikit-image\": \"0.25.2\",\n    \"packaging\": \"25.0\",\n    \"torch\": \"2.11.0+cpu\",\n    \"torchvision\": \"0.26.0+cpu\"\n  },\n  \"environment_boundary\": \"CPU-only private environment; same torch/torchvision release as N08, CPU builds replace cu128. YOLO bitwise GPU reproduction not presumed; saved crop/detection comparison reported.\",\n  \"package_source\": {\n    \"wheel_url\": \"https://files.pythonhosted.org/packages/42/ce/33989427a7fae952e1606d0edee4191f965630f3c74f2166fa776fafd271/insightface-1.0.1-py3-none-any.whl\",\n    \"wheel_sha256\": \"5f373f6fedbdda5cbc59a34ca386a75a2995cdaf6899402590ae9eb4308fc2e8\",\n    \"legacy_url\": \"https://files.pythonhosted.org/packages/0b/8d/0f4af90999ca96cf8cb846eb5ae27c5ef5b390f9c090dd19e4fa76364c13/insightface-0.7.3.tar.gz\",\n    \"legacy_sha256\": \"f191f719612ebb37018f41936814500544cd0f86e6fcd676c023f354c668ddf7\",\n    \"legacy_retinaface_sha256\": \"945597b54141a892419e0edfba464eb84fa2ec10ba800265c06ee274edc534ee\",\n    \"cpu_wheels\": {\n      \"torch\": {\n        \"url\": \"https://download-r2.pytorch.org/whl/cpu/torch-2.11.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl\",\n        \"sha256\": \"f82e2ae20c1545bb03997d1cc3143d94e14b800038669ee1aca45808a9acc338\"\n      },\n      \"torchvision\": {\n        \"url\": \"https://download-r2.pytorch.org/whl/cpu/torchvision-0.26.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl\",\n        \"sha256\": \"cf547dc0975eb40bc3249be4ccbeb736597d2c3ece305b1c4e5b7a5dd7363567\"\n      }\n    }\n  },\n  \"geometry_rules\": {\n    \"min_interocular_px\": 1,\n    \"landmark_box_tolerance_fraction\": 0.25,\n    \"alignment_condition_max\": 100000000,\n    \"warning_padding_fraction\": 0.5,\n    \"automatic_rules\": \"engineering plausibility only; no intended-face acceptance without actual manual review\"\n  },\n  \"privacy_similarity\": \"not_calculated\",\n  \"biometric_threshold\": null,\n  \"prompt\": \"not_applicable_existing_images\",\n  \"mask\": \"none\",\n  \"diffusion_parameters\": \"not_applicable\",\n  \"utility\": \"not_reevaluated\",\n  \"manual_review\": \"pending; never inferred from extraction\",\n  \"cpu_threads\": 2,\n  \"numeric_tolerance\": 0.0001,\n  \"input_contract\": {\n    \"prepared\": \"HWC uint8 BGR 0..255\",\n    \"blob\": \"NCHW float32 RGB (pixel-127.5)/128\",\n    \"pad_value\": 0,\n    \"coordinate_frame\": \"detector.detect returns prepared-input pixel coordinates; crop offsets then map to original\"\n  }\n}")
CONFIG_SHA256 = hashlib.sha256(json.dumps(CONFIG,sort_keys=True,separators=(",",":"),allow_nan=False).encode()).hexdigest()
RUNS_ROOT = Path("/content/drive/MyDrive/thesis_all/outputs/runs")
PARENT07 = RUNS_ROOT / CONFIG["parent07"]
PARENT08 = RUNS_ROOT / CONFIG["parent08"]
WORKER_PARTS = []
print("Frozen configuration:",CONFIG_SHA256,"; 14 sources; CPU diagnostic; no execution switches.")

CONFIG['variants']=[
 {'id':'V0_original_1024','canvas':1024,'kind':'direct','description':'Original full-frame resized by upstream SCRFD to 1024'},
 {'id':'V1_original_640','canvas':640,'kind':'direct','description':'Same source directly, upstream SCRFD resize to 640'},
 {'id':'V2_original_320','canvas':320,'kind':'direct','description':'Same source directly, upstream SCRFD resize to 320'},
 {'id':'V3_centered_640','canvas':640,'kind':'centered','description':'512 original isotropically reduced to 320 and centered on 640 black canvas'},
 {'id':'V4_centered_1024','canvas':1024,'kind':'centered1024','description':'512 original unresized, centered on black 1024 canvas'}]
CONFIG['method_id']='nb10_detector_scale_canvas_v1'
CONFIG['analysis']='detector_localization_only_no_embeddings_or_privacy'
CONFIG_SHA256=hashlib.sha256(json.dumps(CONFIG,sort_keys=True,separators=(',',':')).encode()).hexdigest()
print('NB10 frozen methods:', [(x['id'], x['canvas']) for x in CONFIG['variants']]);print('Config hash:',CONFIG_SHA256)

# N11 pre-registration: freeze V3 only, without adaptive choices.
CONFIG['variants']=[{'id':'V3_centered_640','canvas':640,'kind':'centered','content_size':320,'pad_value':0}]
CONFIG['method_id']='nb11_v3_embedding_validation_v1'
CONFIG['analysis']='120_originals_14_development_106_holdout_no_privacy'
CONFIG['det_thresh']=0.1
CONFIG_SHA256=hashlib.sha256(json.dumps(CONFIG,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()
print('N11 frozen V3; config:',CONFIG_SHA256)

CONFIG['method_id']='nb12_v3_privacy_reevaluation_v1'
CONFIG['analysis']='600_generated_pairs_v3_corrected_biometric_provisional'
CONFIG['variants']=[{'id':'V3_centered_640','canvas':640,'kind':'centered','content_size':320,'pad_value':0}]
CONFIG['det_thresh']=0.1
CONFIG_SHA256=hashlib.sha256(json.dumps(CONFIG,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()
print('N12 scientific config:',CONFIG_SHA256)


## 2 — Automated isolated environment bootstrap
Identical version and model-source verification to the already executed N11; installs only into a private `/content` venv. Uses the non-interactive Matplotlib `Agg` backend.

In [ ]:

if sys.platform != 'linux' or platform.machine() != 'x86_64' or sys.version_info[:2] != (3,12):
    raise RuntimeError('Notebook 10 needs Linux x86_64 CPython 3.12 (prefer 3.12.13). Select a compatible Colab runtime. No pins are substituted.')
libc_name,libc_version=platform.libc_ver()
if libc_name!='glibc' or tuple(map(int,libc_version.split('.')[:2]))<(2,28):
    raise RuntimeError('Pinned CPU wheels require glibc >=2.28; use compatible Colab Python 3.12 runtime.')
if platform.python_version() != '3.12.13':
    print('Runtime patch differs from N08:',platform.python_version(),'; recorded, never silently equated.')
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')

def sha_file(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''): digest.update(block)
    return digest.hexdigest()

PRIVATE_ENV=dict(os.environ)
for name in ['PYTHONPATH','PYTHONHOME','PIP_TARGET','PIP_USER','PIP_PREFIX']:
    PRIVATE_ENV.pop(name,None)
PRIVATE_ENV.update(PIP_CONFIG_FILE=os.devnull,CUDA_VISIBLE_DEVICES='',MPLBACKEND='Agg')

def run_process(command):
    print('Environment step:',command[1:4],flush=True)
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=PRIVATE_ENV)
    for line in process.stdout: print(line.rstrip(),flush=True)
    if process.wait(): raise RuntimeError('Automatic isolated setup failed; no diagnostic inference started. See pip/import error above.')

pin_hash=hashlib.sha256(json.dumps(CONFIG['pins'],sort_keys=True).encode()).hexdigest()[:12]
ENV_DIR=Path('/content')/('nb09_env_'+platform.python_version()+'_'+pin_hash)
ENV_PY=ENV_DIR/'bin'/'python'
PROBE = r'''
import json,sys,platform,hashlib,inspect
from importlib import metadata
import numpy,pandas,PIL,matplotlib,onnx,onnxruntime,insightface,cv2,torch,torchvision,ultralytics
from insightface.app import FaceAnalysis
import insightface.model_zoo.scrfd as scrfd
import insightface.model_zoo.model_zoo as zoo
pins=json.loads(sys.argv[1])
for name,expected in pins.items():
    if metadata.version(name)!=expected: raise RuntimeError(f'Pin mismatch: {name}, expected {expected}')
if numpy.__version__!=pins['numpy'] or cv2.__version__!='.'.join(pins['opencv-python'].split('.')[:3]):
    raise RuntimeError('Native import/version collision')
if 'CPUExecutionProvider' not in onnxruntime.get_available_providers(): raise RuntimeError('CPU ONNX unavailable')
result={'python':platform.python_version(),'platform':platform.platform(),'packages':dict(sorted((d.metadata['Name'],d.version) for d in metadata.distributions())),
 'torch_device':str(torch.device('cpu')),'torch':torch.__version__,'torchvision':torchvision.__version__,
 'onnx_providers':onnxruntime.get_available_providers(),'source_sha256':{}}
for obj in [FaceAnalysis,scrfd.SCRFD,zoo.ModelRouter]:
    path=inspect.getsourcefile(obj)
    result['source_sha256'][obj.__name__]=hashlib.sha256(open(path,'rb').read()).hexdigest()
print('NB09_ENV '+json.dumps(result,sort_keys=True))
'''
# Colab may ship a Python interpreter whose ensurepip bootstrap cannot complete.
# Create a pip-free venv, then bootstrap pinned pip *inside that venv*
# using host pip's --python target (does not install packages into the host).
import venv
import shutil
PIP_BOOTSTRAP_VERSION = '24.3.1'

def private_pip_works():
    if not ENV_PY.is_file():
        return False
    try:
        check = subprocess.run(
            [str(ENV_PY), '-m', 'pip', '--version'],
            env=PRIVATE_ENV, text=True, capture_output=True, timeout=30,
        )
        return check.returncode == 0 and str(ENV_DIR) in check.stdout
    except (OSError, subprocess.TimeoutExpired):
        return False

if not private_pip_works():
    # Only the uniquely named, ephemeral /content/nb09_env_* directory is touched.
    if ENV_DIR.exists():
        if ENV_DIR.parent != Path('/content') or not ENV_DIR.name.startswith('nb09_env_'):
            raise RuntimeError('Unsafe private environment path; refusing cleanup')
        print('Cleaning incomplete Notebook 10 venv:', ENV_DIR, flush=True)
        shutil.rmtree(ENV_DIR)
    try:
        venv.EnvBuilder(with_pip=False, system_site_packages=False).create(ENV_DIR)
    except Exception as error:
        raise RuntimeError('Unable to create pip-free isolated venv: ' + repr(error)) from error
    # pip >=22.3 supports --python and can provision pip into a venv with no pip.
    host_pip = subprocess.run(
        [sys.executable, '-m', 'pip', '--version'],
        env=PRIVATE_ENV, text=True, capture_output=True, timeout=30,
    )
    if host_pip.returncode:
        raise RuntimeError('Host pip unavailable for target-only bootstrap: ' + host_pip.stderr[-3000:])
    run_process([
        sys.executable, '-m', 'pip', '--python', str(ENV_PY),
        '--isolated', 'install', '--disable-pip-version-check',
        '--index-url', 'https://pypi.org/simple',
        'pip==' + PIP_BOOTSTRAP_VERSION,
    ])
    if not private_pip_works():
        raise RuntimeError('Pinned pip bootstrap did not produce working private pip; no inference started')
VERSION_PROBE="import json,sys;from importlib.metadata import version; pins=json.loads(sys.argv[1]);assert all(version(k)==v for k,v in pins.items())"
versions_ready=subprocess.run([str(ENV_PY),'-c',VERSION_PROBE,json.dumps(CONFIG['pins'])],capture_output=True,env=PRIVATE_ENV).returncode==0
if not versions_ready:
    requirements=[name+'=='+version for name,version in CONFIG['pins'].items()]
    requirements=[value for value in requirements if not value.startswith(('insightface==','torch==','torchvision=='))]
    requirements += [spec['url']+'#sha256='+spec['sha256'] for spec in CONFIG['package_source']['cpu_wheels'].values()]
    requirements.append(CONFIG['package_source']['wheel_url']+'#sha256='+CONFIG['package_source']['wheel_sha256'])
    run_process([str(ENV_PY),'-m','pip','--isolated','install','--timeout','30','--retries','2',
                 '--index-url','https://pypi.org/simple',*requirements])
run_process([str(ENV_PY),'-m','pip','--isolated','check'])
probe_env=dict(PRIVATE_ENV,CUDA_VISIBLE_DEVICES='',YOLO_CONFIG_DIR=str(ENV_DIR/'yolo_settings'),MPLCONFIGDIR=str(ENV_DIR/'mpl'))
probe=subprocess.run([str(ENV_PY),'-c',PROBE,json.dumps(CONFIG['pins'])],env=probe_env,text=True,capture_output=True,timeout=180)
if probe.returncode: raise RuntimeError('Private environment import/ABI/pin preflight failed before inference: '+probe.stderr[-6000:])
probe_lines=[line[9:] for line in probe.stdout.splitlines() if line.startswith('NB09_ENV ')]
if len(probe_lines)!=1: raise RuntimeError('Environment probe missing structured output')
ENV_METADATA=json.loads(probe_lines[0])
EXPECTED_PACKAGE_SOURCE={'FaceAnalysis':'759271af722dafa1f602354550cdfd84cd705de0184a63b43752982635bcf5ff',
                         'SCRFD':'8898d1f90690d46bee371e1de31d1af60837988272a5eef904c5ba89e0fb8d83'}
for name,expected in EXPECTED_PACKAGE_SOURCE.items():
    if ENV_METADATA['source_sha256'][name]!=expected:raise RuntimeError('Installed official InsightFace source mismatch: '+name)
ENV_SHA256=hashlib.sha256(json.dumps(ENV_METADATA,sort_keys=True,separators=(',',':')).encode()).hexdigest()
print('Isolated environment verified; no restart/model loading. Fingerprint:',ENV_SHA256)


## 3 — Immutable data and model preflight
Assert **exactly 120** originals and **600** successful unique generated-image records. Read N07 generation CSV and row-level evaluation CSV; never infer filenames or silently drop failures. Preserve original paths, image hashes, condition, seed, and tuning-source labels.

In [ ]:
from PIL import Image
NB09_RUN=RUNS_ROOT/'20261013_nb09_detector-rootcause_4fae269d_3a2979d8_20261013T201108057643Z_02a058fe'
NB09_SESSION=NB09_RUN/'sessions'/'20261013T201108057643Z_02a058fe'
NB09_HASHES=NB09_SESSION/'model_file_hashes.json'
NB10_RUN=RUNS_ROOT/'nb10_scale_canvas_c932b62da2be_20261013T203150121955Z_5366f3bc'
NB11_RUN=RUNS_ROOT/'nb11_v3_alignment_d4858ad3aaea_20261013T204443805466Z_1b4f5acf'
PARENT07=RUNS_ROOT/CONFIG['parent07']
for required in [NB09_HASHES,NB10_RUN/'summary.json',NB11_RUN/'summary.json',PARENT07/'generation_results.csv',PARENT07/'raw_results.csv',PARENT07/'selected_sources.csv']:
    if not required.is_file():raise FileNotFoundError('Missing immutable historic evidence: '+str(required))
nb11_summary=json.loads((NB11_RUN/'summary.json').read_text())
assert nb11_summary['records']==120 and nb11_summary['subsets']['heldout_106']['n']==106
archive=json.loads(NB09_HASHES.read_text())
expected={'detector':CONFIG['models']['arcface']['onnx_sha256']['det_10g.onnx'],
 'arcface':CONFIG['models']['arcface']['onnx_sha256']['w600k_r50.onnx'],
 'auraface':CONFIG['models']['auraface']['onnx_sha256']['glintr100.onnx']}
resolved={}
for name,sha in expected.items():
    candidates=[Path(p) for p,h in archive.items() if h==sha and Path(p).is_file()]
    if not candidates:raise FileNotFoundError('N09 SHA-verified weight not present: '+name)
    p=candidates[0]
    if sha_file(p)!=sha:raise RuntimeError('Model hash changed: '+name)
    resolved[name]=str(p)
source_dir=PARENT07/'sources'
source_paths=sorted(source_dir.glob('source_*.png'))
if len(source_paths)!=120:raise RuntimeError('Expected 120 N07 source PNGs')
sources=[{'source_id':p.stem,'path':str(p),'sha256':sha_file(p)} for p in source_paths]
assert len(set(r['source_id'] for r in sources))==120
private_manifest=PARENT07/'metadata'/'private_development_source_hashes.json'
if not private_manifest.is_file(): raise RuntimeError('Private verification manifest missing')
expected_hashes=json.loads(private_manifest.read_text())
frozen14=expected_hashes
for r in sources:
    if r['source_id'] in frozen14 and r['sha256']!=frozen14[r['source_id']]:raise RuntimeError('Development source checksum mismatch: '+r['source_id'])
source_ids=set(r['source_id'] for r in sources)
with (PARENT07/'generation_results.csv').open(newline='') as f:gen=list(csv.DictReader(f))
if len(gen)!=600:raise RuntimeError(f'Expected 600 generated records, got {len(gen)}')
with (PARENT07/'raw_results.csv').open(newline='') as f:old=list(csv.DictReader(f))
keys=['source_id','condition','seed']
assert all(k in gen[0] for k in keys) and all(k in old[0] for k in keys)
def rowkey(r):return tuple(str(r[k]).strip() for k in keys)
old_by_key={rowkey(r):r for r in old}
if len(old_by_key)!=len(old):raise RuntimeError('Duplicate N07 raw evaluation keys')
if len({rowkey(r) for r in gen})!=600 or len(old_by_key)!=600:raise RuntimeError('Duplicate or absent experiment keys')
if set(rowkey(r) for r in gen)!=set(old_by_key):raise RuntimeError('N07 generation/evaluation records differ')
outputs=[]
for row in gen:
    key=rowkey(row)
    if row['source_id'] not in source_ids:raise RuntimeError('Orphan generated output')
    if row['generation_status'].strip().lower()!='success':raise RuntimeError('Expected 600 successful generations; do not silently omit errors')
    path=Path(row['output_path'])
    if not path.is_file() or not path.is_relative_to(PARENT07/'images'):raise RuntimeError('Missing or out-of-scope generated image: '+str(path))
    with Image.open(path) as im:
        if im.size!=(512,512):raise RuntimeError('Unexpected generated image dimensions: '+str(path))
    outputs.append({'source_id':row['source_id'],'condition':row['condition'],'condition_family':old_by_key[key].get('condition_family',''),
      'condition_role':old_by_key[key].get('condition_role',''),'guidance':old_by_key[key].get('guidance',''),
      'seed':row['seed'],'path':str(path),'sha256':sha_file(path),'historical':{
       k:old_by_key[key].get(k,'') for k in ['human_target_emotion','emotion_preserved','emotion_status','original_predicted_class',
       'output_predicted_class','absolute_target_probability_delta','emotion_distribution_cosine',
       'arcface_similarity','auraface_similarity','arcface_status','auraface_status']}})
assert len(outputs)==600
run_id='nb12_v3_biometric_'+CONFIG_SHA256[:12]+'_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')+'_'+uuid.uuid4().hex[:8]
OUT_DIR=RUNS_ROOT/run_id
OUT_DIR.mkdir(exist_ok=False)
context={'run_id':run_id,'scientific_config':CONFIG,'config_sha256':CONFIG_SHA256,
 'models':resolved,'model_sha256':expected,'sources':sources,'outputs':outputs,
 'historical_artifacts':{'generation_results_sha256':sha_file(PARENT07/'generation_results.csv'),
  'raw_results_sha256':sha_file(PARENT07/'raw_results.csv'),'nb11_summary_sha256':sha_file(NB11_RUN/'summary.json')},
 'manual_review':'pending','identity_validity':'not_yet_validated','reidentification_threshold':None}
(OUT_DIR/'context.json').write_text(json.dumps(context,indent=2,allow_nan=False))
print('Preflight OK:',len(sources),'originals and',len(outputs),'generated; outputs:',OUT_DIR)


## 4 — Corrected biometric extraction and provisional cosine scoring
Extract originals **once** and each generated image once. Source reference and generated faces share exactly the same V3 detector and alignment; finite 512D embeddings are required for cosine. Every failure is an explicit row, never zero similarity. Write per-image immutable checkpoints and contact-sheet previews for independent manual review.

In [ ]:
WORKER_SOURCE = 'import sys,os,json,csv,hashlib,traceback,time,math\nfrom pathlib import Path\nimport numpy as np, cv2\nfrom PIL import Image,ImageDraw\nfrom insightface.model_zoo import get_model\nfrom insightface.utils import face_align\nctx=json.loads(Path(sys.argv[1]).read_text()); root=Path(sys.argv[2]);config=ctx[\'scientific_config\']\n(root/\'checkpoints\').mkdir(exist_ok=True);(root/\'previews\').mkdir(exist_ok=True)\ndef digest(p):\n h=hashlib.sha256()\n with open(p,\'rb\') as f:\n  for b in iter(lambda:f.read(1048576),b\'\'):h.update(b)\n return h.hexdigest()\ndef atomic_json(p,d):\n tmp=p.with_suffix(p.suffix+\'.partial\')\n with open(tmp,\'x\') as f:json.dump(d,f,allow_nan=False,indent=2)\n os.replace(tmp,p)\ndef finite(v):\n f=float(v);return f if math.isfinite(f) else None\nfor name,p in ctx[\'models\'].items():\n if digest(p)!=ctx[\'model_sha256\'][name]:raise RuntimeError(\'Model changed: \'+name)\ndet=get_model(ctx[\'models\'][\'detector\'],providers=[\'CPUExecutionProvider\']);assert type(det).__name__==\'SCRFD\'\ndet.prepare(ctx_id=-1,input_size=(640,640),det_thresh=0.1)\nrecognizers={}\nfor name in (\'arcface\',\'auraface\'):\n rec=get_model(ctx[\'models\'][name],providers=[\'CPUExecutionProvider\']);rec.prepare(ctx_id=-1);recognizers[name]=rec\nnp.random.seed(42)\ndef extract(item,tag):\n sid=item[\'source_id\'];status={\'source_id\':sid,\'image_sha256\':item[\'sha256\'],\'num_detections\':0,\n \'selected_bbox\':None,\'landmarks\':None,\'detection_confidence\':None,\'box_intersection_fraction\':None,\n \'alignment_plausible\':False,\'manual_face_acceptance\':\'pending\',\'error\':None,\'recognizers\':{}}\n embeds={}; preview=Image.new(\'RGB\',(768,384),\'white\')\n try:\n  if digest(item[\'path\'])!=item[\'sha256\']:raise RuntimeError(\'Image SHA changed\')\n  rgb=np.asarray(Image.open(item[\'path\']).convert(\'RGB\'))\n  if rgb.shape!=(512,512,3):raise RuntimeError(\'Expected HWC RGB 512x512\')\n  bgr=cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR)\n  inp=np.zeros((640,640,3),dtype=np.uint8)\n  inp[160:480,160:480]=cv2.resize(bgr,(320,320),interpolation=cv2.INTER_AREA)\n  boxes,lm=det.detect(inp,max_num=0,metric=\'default\')\n  status[\'num_detections\']=int(len(boxes))\n  original=Image.fromarray(rgb).resize((384,384));draw=ImageDraw.Draw(original)\n  if len(boxes):\n   # Frozen deterministic selection: top-confidence SCRFD result. No YOLO backup, no threshold tuning.\n   ix=int(np.argmax(boxes[:,4])); b=np.asarray(boxes[ix],dtype=np.float64);pts=(np.asarray(lm[ix],dtype=np.float64)-160)/0.625\n   xy=(b[:4]-160)/0.625;status[\'selected_bbox\']=[finite(v) for v in xy]\n   status[\'landmarks\']=[[finite(z) for z in pair] for pair in pts]\n   status[\'detection_confidence\']=finite(b[4]);x1,y1,x2,y2=xy\n   area=max(0,x2-x1)*max(0,y2-y1)\n   overlap=max(0,min(512,x2)-max(0,x1))*max(0,min(512,y2)-max(0,y1))\n   status[\'box_intersection_fraction\']=finite(overlap/area) if area>0 else 0.\n   status[\'landmarks_plausible\']=bool(pts.shape==(5,2) and np.isfinite(pts).all() and np.linalg.norm(pts[0]-pts[1])>=1 and status[\'box_intersection_fraction\']>=0.7)\n   aligned=face_align.norm_crop(bgr,landmark=pts.astype(np.float32),image_size=112)\n   status[\'alignment_plausible\']=bool(aligned.shape==(112,112,3) and aligned.dtype==np.uint8 and status[\'landmarks_plausible\'])\n   draw.rectangle([float(v)*0.75 for v in xy],outline=\'yellow\',width=3)\n   for x,y in pts:draw.ellipse(((x-3)*.75,(y-3)*.75,(x+3)*.75,(y+3)*.75),fill=\'red\')\n   aligned_image=Image.fromarray(cv2.cvtColor(aligned,cv2.COLOR_BGR2RGB)).resize((336,336),Image.Resampling.NEAREST)\n   preview.paste(aligned_image,(408,24))\n   for name,rec in recognizers.items():\n    try:\n     arr=np.asarray(rec.get_feat(aligned),dtype=np.float32).reshape(-1)\n     norm=float(np.linalg.norm(arr));valid=(arr.size==512 and np.isfinite(arr).all() and np.isfinite(norm) and norm>1e-10 and status[\'alignment_plausible\'])\n     status[\'recognizers\'][name]={\'embedding_extracted\':bool(valid),\'dimension\':int(arr.size),\'norm\':finite(norm)}\n     if valid:embeds[name]=arr/norm\n    except Exception as e:status[\'recognizers\'][name]={\'embedding_extracted\':False,\'error\':repr(e)}\n  else:\n   for name in recognizers:status[\'recognizers\'][name]={\'embedding_extracted\':False,\'reason\':\'no_face\'}\n  preview.paste(original,(0,0))\n except Exception as e:\n  status[\'error\']=type(e).__name__+\': \'+str(e)\n  status[\'traceback\']=traceback.format_exc()\n preview.save(root/\'previews\'/(tag+\'.png\'))\n return status,embeds\ndef key_of(o):return o[\'source_id\']+\'__\'+o[\'condition\']+\'__seed\'+str(o[\'seed\'])\nsources={}\nprint(\'Extracting 120 sources for reference...\',flush=True)\nfor i,item in enumerate(ctx[\'sources\']):\n sid=item[\'source_id\'];status,emb=extract(item,\'original_\'+sid)\n sources[sid]=(status,emb)\n atomic_json(root/\'checkpoints\'/(\'original_\'+sid+\'.json\'),status)\n if (i+1)%20==0:print(\'Sources\',i+1,\'/120\',flush=True)\nrecords=[]\nfor i,item in enumerate(ctx[\'outputs\']):\n tag=key_of(item);out=root/\'checkpoints\'/(tag+\'.json\')\n if out.exists():\n  r=json.loads(out.read_text())\n  if r.get(\'output_sha256\')!=item[\'sha256\'] or r.get(\'config_sha256\')!=ctx[\'config_sha256\']:raise RuntimeError(\'Checkpoint incompatible: \'+tag)\n  records.append(r);continue\n src,emb_src=sources[item[\'source_id\']]\n generated,emb_gen=extract(item,tag)\n r={\'source_id\':item[\'source_id\'],\'condition\':item[\'condition\'],\'condition_family\':item[\'condition_family\'],\n \'condition_role\':item[\'condition_role\'],\'guidance\':item[\'guidance\'],\'seed\':item[\'seed\'],\n \'output_sha256\':item[\'sha256\'],\'source_sha256\':src[\'image_sha256\'],\'config_sha256\':ctx[\'config_sha256\'],\n \'source_detection\':src,\'generated_detection\':generated,\'historical_n07\':item[\'historical\'],\n \'manual_pair_acceptance\':\'pending\',\'arcface_cosine\':None,\'auraface_cosine\':None,\'pair_valid_arcface\':False,\'pair_valid_auraface\':False}\n for name in recognizers:\n  if name in emb_src and name in emb_gen:\n   cosine=float(np.dot(emb_src[name],emb_gen[name]));r[name+\'_cosine\']=finite(cosine)\n   r[\'pair_valid_\'+name]=bool(np.isfinite(cosine))\n atomic_json(out,r);records.append(r)\n if (i+1)%25==0 or i==0:print(\'Generated\',i+1,\'/600; pair ok ArcFace\',sum(z[\'pair_valid_arcface\'] for z in records),\'AuraFace\',sum(z[\'pair_valid_auraface\'] for z in records),flush=True)\nassert len(records)==600 and len(set((r[\'source_id\'],r[\'condition\'],r[\'seed\']) for r in records))==600\ncols=[\'source_id\',\'condition\',\'condition_family\',\'condition_role\',\'guidance\',\'seed\',\'source_sha256\',\'output_sha256\',\n \'source_detected\',\'output_detected\',\'source_alignment_plausible\',\'output_alignment_plausible\',\n \'source_manual_review\',\'output_manual_review\',\'pair_manual_review\',\'arcface_cosine\',\'auraface_cosine\',\'pair_valid_arcface\',\'pair_valid_auraface\',\n \'historical_emotion_status\',\'historical_emotion_preserved\',\'historical_arcface_similarity\',\'historical_auraface_similarity\',\'output_error\']\nwith open(root/\'raw_corrected_privacy.csv\',\'w\',newline=\'\') as f:\n w=csv.DictWriter(f,fieldnames=cols);w.writeheader()\n for r in records:\n  src=r[\'source_detection\'];out=r[\'generated_detection\'];h=r[\'historical_n07\']\n  w.writerow({\'source_id\':r[\'source_id\'],\'condition\':r[\'condition\'],\'condition_family\':r[\'condition_family\'],\n   \'condition_role\':r[\'condition_role\'],\'guidance\':r[\'guidance\'],\'seed\':r[\'seed\'],\'source_sha256\':r[\'source_sha256\'],\n   \'output_sha256\':r[\'output_sha256\'],\'source_detected\':bool(src[\'num_detections\']),\n   \'output_detected\':bool(out[\'num_detections\']),\'source_alignment_plausible\':src[\'alignment_plausible\'],\n   \'output_alignment_plausible\':out[\'alignment_plausible\'],\'source_manual_review\':\'pending\',\'output_manual_review\':\'pending\',\n   \'pair_manual_review\':\'pending\',\'arcface_cosine\':r[\'arcface_cosine\'],\'auraface_cosine\':r[\'auraface_cosine\'],\n   \'pair_valid_arcface\':r[\'pair_valid_arcface\'],\'pair_valid_auraface\':r[\'pair_valid_auraface\'],\n   \'historical_emotion_status\':h[\'emotion_status\'],\'historical_emotion_preserved\':h[\'emotion_preserved\'],\n   \'historical_arcface_similarity\':h[\'arcface_similarity\'],\'historical_auraface_similarity\':h[\'auraface_similarity\'],\n   \'output_error\':out[\'error\']})\nsummary={\'status\':\'completed_pending_manual_review\',\'records\':len(records),\'sources\':len(sources),\'config_sha256\':ctx[\'config_sha256\'],\n \'valid_reference_arcface\':sum(\'arcface\' in v[1] for v in sources.values()),\n \'valid_reference_auraface\':sum(\'auraface\' in v[1] for v in sources.values()),\n \'generated_detection_count\':sum(r[\'generated_detection\'][\'num_detections\']>0 for r in records),\n \'pairs_finite_arcface\':sum(r[\'pair_valid_arcface\'] for r in records),\n \'pairs_finite_auraface\':sum(r[\'pair_valid_auraface\'] for r in records),\n \'manual_pair_acceptance\':\'pending\',\'validated_privacy_conclusion\':None,\'reidentification_risk\':None,\n \'by_condition\':{}}\nfor condition in sorted({r[\'condition\'] for r in records}):\n rr=[r for r in records if r[\'condition\']==condition]\n summary[\'by_condition\'][condition]={\'n\':len(rr)}\n for name in (\'arcface\',\'auraface\'):\n  valid=[r[name+\'_cosine\'] for r in rr if r[\'pair_valid_\'+name]]\n  summary[\'by_condition\'][condition][name]={\'finite_pair_count\':len(valid),\'mean_cosine\':float(np.mean(valid)) if valid else None,\n   \'median_cosine\':float(np.median(valid)) if valid else None,\'std_cosine\':float(np.std(valid,ddof=1)) if len(valid)>1 else None}\natomic_json(root/\'summary.json\',summary)\n# Visual review: source reference + generated face previews arranged in pages, not auto approval.\nfor page,start in enumerate(range(0,len(ctx[\'outputs\']),24),1):\n batch=ctx[\'outputs\'][start:start+24]; sheet=Image.new(\'RGB\',(4*400,6*225),\'white\');d=ImageDraw.Draw(sheet)\n for j,item in enumerate(batch):\n  p=root/\'previews\'/(key_of(item)+\'.png\');im=Image.open(p).convert(\'RGB\').resize((392,196))\n  x=(j%4)*400;y=(j//4)*225;sheet.paste(im,(x,y+20));d.text((x+3,y+2),key_of(item)[:46],fill=\'black\')\n sheet.save(root/(f\'contact_generated_page_{page:02d}.jpg\'),quality=85)\nreport=[\'# Notebook 12 — Corrected V3 privacy re-evaluation\',\'\',\n \'**Provisional detector/embedding results; manual assignment/alignment review pending.**\',\'\',\n f"Completed {len(records)} generated pairs against {len(sources)} originals.",\n f"Finite ArcFace pairs: {summary[\'pairs_finite_arcface\']}/600; AuraFace: {summary[\'pairs_finite_auraface\']}/600.",\n f"Generated face detections: {summary[\'generated_detection_count\']}/600.",\'\',\n \'## Conditional provisional results\',\'\',\n \'| Condition | N | ArcFace valid | ArcFace mean cosine | AuraFace valid | AuraFace mean cosine |\',\n \'|---|---:|---:|---:|---:|---:|\']\nfor cond,s in summary[\'by_condition\'].items():\n a=s[\'arcface\'];b=s[\'auraface\'];fmt=lambda x:f\'{x:.4f}\' if x is not None else \'NA\'\n report.append(f"| {cond} | {s[\'n\']} | {a[\'finite_pair_count\']} | {fmt(a[\'mean_cosine\'])} | {b[\'finite_pair_count\']} | {fmt(b[\'mean_cosine\'])} |")\nreport+=[\'\',\'**Do not treat lower similarity as proven anonymity or report re-identification rates.**\',\n \'Failures are missing outcomes and remain in the denominator. The original N07 emotion utility was not rerun.\',\n \'Automatic geometric checks are NOT ground truth; inspect previews before drawing privacy conclusions.\',\n \'N07 historical biometric scores appear only for comparison and have not been overwritten.\',\n \'The 14 original development images are reused; sensitivity analysis excluding 12 tuning sources requires explicit identification from N07.\']\n(root/\'diagnostic_report.md\').write_text(\'\\n\'.join(report))\nprint(\'NB12_COMPLETED\',str(root),flush=True)\n'
WORKER_PATH=OUT_DIR/'worker.py'
WORKER_PATH.write_text(WORKER_SOURCE)
process=subprocess.Popen([str(ENV_PY),'-u',str(WORKER_PATH),str(OUT_DIR/'context.json'),str(OUT_DIR)],
 stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=probe_env)
for line in process.stdout: print(line.rstrip(),flush=True)
if process.wait()!=0:raise RuntimeError('N12 worker failed; historical outputs untouched. Inspect last log and partial checkpoints in new output directory.')


## 5 — Evidence check, review gate, and interpretation boundary
Outputs: `raw_corrected_privacy.csv`, 600 per-pair JSON checkpoints, 120 source previews, 600 generated previews, contact-sheet JPGs, `summary.json` and `diagnostic_report.md`. Technical similarity is **not** validated identity privacy; all manual reviews remain pending. No face-match threshold is selected and no re-identification rate is claimed. For statistically defensible reporting, review the contact sheets and conduct pre-specified paired/cluster-aware analyses and tuning-source exclusions separately.

In [ ]:
summary=json.loads((OUT_DIR/'summary.json').read_text())
assert summary['records']==600 and summary['sources']==120
assert summary['manual_pair_acceptance']=='pending' and summary['validated_privacy_conclusion'] is None
print((OUT_DIR/'diagnostic_report.md').read_text())
print('OUTPUT FOLDER:',OUT_DIR)
